# Walmart sales cleaning

Raw Walmart.csv is preserved. Dates are interpreted month-first. Configure WALMART_SQL_URL only if you intentionally want to replace a practice SQL table; normal notebook execution exports a local cleaned CSV only.

In [1]:
from pathlib import Path
import pandas as pd
df=pd.read_csv("Walmart.csv")
print("Raw rows:",len(df))
df=df.drop_duplicates().dropna(subset=["unit_price","quantity","date"]).copy()
df["unit_price"]=pd.to_numeric(df["unit_price"].astype(str).str.replace("$","",regex=False),errors="raise")
df["date"]=pd.to_datetime(df["date"],errors="raise",format="mixed",dayfirst=False)
df["Total"]=df["quantity"]*df["unit_price"]
Path("reports").mkdir(exist_ok=True)
df.to_csv("reports/Walmart_cleaned.csv",index=False)
print("Cleaned rows:",len(df),"Total sales:",df["Total"].sum())

Raw rows: 9969
Cleaned rows: 9969 Total sales: 1209726.38


In [2]:
import os
url=os.getenv("WALMART_SQL_URL")
if url:
    from sqlalchemy import create_engine
    with create_engine(url).begin() as connection:
        df.to_sql("Walmart",connection,if_exists="replace",index=False)
    print("Practice SQL table replaced as explicitly configured.")
else:
    print("SQL export skipped; WALMART_SQL_URL is unset.")

SQL export skipped; WALMART_SQL_URL is unset.


SQL now ranks payment methods within each branch, preserving ties. The revenue-drop query measures absolute amount, not percentage decline. SQL Server validation requires an owner-configured practice database.